In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
from calibration import calibration_error
from sklearn.calibration import CalibratedClassifierCV
from joblib import dump

In [2]:
swing_train = pd.read_parquet(
    "https://lab.cs307.org/swing/data/swing-train.parquet",
)
swing_test = pd.read_parquet(
    "https://lab.cs307.org/swing/data/swing-test.parquet",
)

# create X and y for train
X_train = swing_train.drop("swing", axis=1)
y_train = swing_train["swing"]

# create X and y for test
X_test = swing_test.drop("swing", axis=1)
y_test = swing_test["swing"]

In [8]:
swing_train

,pitch_name,release_extension,release_pos_x,release_pos_y,release_pos_z,release_speed,release_spin_rate,spin_axis,plate_x,plate_z,...,balls,strikes,on_3b,on_2b,on_1b,outs_when_up,stand,sz_top,sz_bot,swing
0,Cutter,6.6,-2.76,53.86,5.81,92.6,2376.0,195.0,-0.09,2.79,...,3,1,0,0,0,1,L,3.15,1.52,1
1,Changeup,6.8,-2.87,53.74,5.66,86.3,1511.0,226.0,-1.47,1.84,...,2,1,0,0,0,1,L,3.13,1.56,0
2,Changeup,6.7,-2.83,53.82,5.68,87.9,1570.0,224.0,-1.52,2.38,...,1,1,0,0,0,1,L,3.12,1.51,0
3,Knuckle Curve,6.7,-2.70,53.78,5.78,82.4,2398.0,32.0,0.20,1.04,...,1,0,0,0,0,1,L,3.15,1.52,1
4,Cutter,6.7,-2.64,53.83,5.81,91.0,2427.0,189.0,0.89,1.65,...,0,0,0,0,0,1,L,3.12,1.51,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2658,4-Seam Fastball,6.8,-2.64,53.75,5.99,93.4,2411.0,206.0,0.59,2.91,...,0,0,0,0,0,1,L,3.50,1.81,0
2659,Slider,6.3,-2.91,54.19,5.86,87.3,2541.0,113.0,1.38,1.73,...,1,2,0,0,0,0,R,3.19,1.48,1
2660,Knuckle Curve,6.4,-2.91,54.13,5.87,84.7,2539.0,35.0,0.81,0.11,...,1,1,0,0,0,0,R,3.19,1.48,1
2661,4-Seam Fastball,6.4,-2.67,54.13,6.04,94.3,2531.0,201.0,1.05,1.79,...,0,1,0,0,0,0,R,3.03,1.48,0


In [5]:
swing_train

,pitch_name,release_extension,release_pos_x,release_pos_y,release_pos_z,release_speed,release_spin_rate,spin_axis,plate_x,plate_z,...,balls,strikes,on_3b,on_2b,on_1b,outs_when_up,stand,sz_top,sz_bot,swing
0,Cutter,6.6,-2.76,53.86,5.81,92.6,2376.0,195.0,-0.09,2.79,...,3,1,0,0,0,1,L,3.15,1.52,1
1,Changeup,6.8,-2.87,53.74,5.66,86.3,1511.0,226.0,-1.47,1.84,...,2,1,0,0,0,1,L,3.13,1.56,0
2,Changeup,6.7,-2.83,53.82,5.68,87.9,1570.0,224.0,-1.52,2.38,...,1,1,0,0,0,1,L,3.12,1.51,0
3,Knuckle Curve,6.7,-2.70,53.78,5.78,82.4,2398.0,32.0,0.20,1.04,...,1,0,0,0,0,1,L,3.15,1.52,1
4,Cutter,6.7,-2.64,53.83,5.81,91.0,2427.0,189.0,0.89,1.65,...,0,0,0,0,0,1,L,3.12,1.51,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2658,4-Seam Fastball,6.8,-2.64,53.75,5.99,93.4,2411.0,206.0,0.59,2.91,...,0,0,0,0,0,1,L,3.50,1.81,0
2659,Slider,6.3,-2.91,54.19,5.86,87.3,2541.0,113.0,1.38,1.73,...,1,2,0,0,0,0,R,3.19,1.48,1
2660,Knuckle Curve,6.4,-2.91,54.13,5.87,84.7,2539.0,35.0,0.81,0.11,...,1,1,0,0,0,0,R,3.19,1.48,1
2661,4-Seam Fastball,6.4,-2.67,54.13,6.04,94.3,2531.0,201.0,1.05,1.79,...,0,1,0,0,0,0,R,3.03,1.48,0


In [6]:
X_train

,pitch_name,release_extension,release_pos_x,release_pos_y,release_pos_z,release_speed,release_spin_rate,spin_axis,plate_x,plate_z,...,pfx_z,balls,strikes,on_3b,on_2b,on_1b,outs_when_up,stand,sz_top,sz_bot
0,Cutter,6.6,-2.76,53.86,5.81,92.6,2376.0,195.0,-0.09,2.79,...,0.97,3,1,0,0,0,1,L,3.15,1.52
1,Changeup,6.8,-2.87,53.74,5.66,86.3,1511.0,226.0,-1.47,1.84,...,0.40,2,1,0,0,0,1,L,3.13,1.56
2,Changeup,6.7,-2.83,53.82,5.68,87.9,1570.0,224.0,-1.52,2.38,...,0.84,1,1,0,0,0,1,L,3.12,1.51
3,Knuckle Curve,6.7,-2.70,53.78,5.78,82.4,2398.0,32.0,0.20,1.04,...,-0.91,1,0,0,0,0,1,L,3.15,1.52
4,Cutter,6.7,-2.64,53.83,5.81,91.0,2427.0,189.0,0.89,1.65,...,1.00,0,0,0,0,0,1,L,3.12,1.51
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2658,4-Seam Fastball,6.8,-2.64,53.75,5.99,93.4,2411.0,206.0,0.59,2.91,...,1.45,0,0,0,0,0,1,L,3.50,1.81
2659,Slider,6.3,-2.91,54.19,5.86,87.3,2541.0,113.0,1.38,1.73,...,-0.01,1,2,0,0,0,0,R,3.19,1.48
2660,Knuckle Curve,6.4,-2.91,54.13,5.87,84.7,2539.0,35.0,0.81,0.11,...,-0.97,1,1,0,0,0,0,R,3.19,1.48
2661,4-Seam Fastball,6.4,-2.67,54.13,6.04,94.3,2531.0,201.0,1.05,1.79,...,1.52,0,1,0,0,0,0,R,3.03,1.48


In [4]:
p_all = swing_train['swing'].mean()
p_type = pd.DataFrame(swing_train.groupby('pitch_name')['swing'].mean())

# print("All:", p_all)
# print(p_type)
p_type

,swing
pitch_name,
4-Seam Fastball,0.470904
Changeup,0.536313
Cutter,0.451477
Knuckle Curve,0.462875
Slider,0.496124


Test ECE: 0.065

Test MCE: 0.12

Test Brier Score: 0.19

Production ECE: 0.065

Production MCE: 0.12

Production Brier Score: 0.19

In [6]:
numeric_transformer = make_pipeline(SimpleImputer(strategy = 'mean'), StandardScaler())
categorical_transformer = make_pipeline(SimpleImputer(strategy = "most_frequent"), OneHotEncoder())

numeric_features = X_train.select_dtypes(include = ['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include = ['object', 'category']).columns

preprocessor = make_column_transformer((numeric_transformer, numeric_features), (categorical_transformer, categorical_features))
preprocessor.fit(X_train)
preprocessor.transform(X_train)

array([[ 0.18450613,  0.30460719, -0.38870632, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.18621828, -0.40518984, -0.99448535, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.6853622 , -0.14708183, -0.59063266, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.81720603, -0.66329785,  0.97429651, ...,  0.        ,
         0.        ,  1.        ],
       [-0.81720603,  0.88535022,  0.97429651, ...,  0.        ,
         0.        ,  1.        ],
       [-1.31806211, -0.34066284,  1.27718603, ...,  0.        ,
         0.        ,  1.        ]], shape=(2663, 26))

In [21]:
numeric_transformer = make_pipeline(SimpleImputer(strategy = 'mean'), StandardScaler())
categorical_transformer = make_pipeline(SimpleImputer(strategy = "most_frequent"), OneHotEncoder())
numeric_features = X_train.select_dtypes(include = ['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include = ['object', 'category']).columns

preprocessor = make_column_transformer((numeric_transformer, numeric_features), (categorical_transformer, categorical_features))
preprocessor.fit(X_train)
preprocessor.transform(X_train)

base_tree = DecisionTreeClassifier(max_depth = 5, min_samples_leaf = 10, random_state = 42)
calibrated_tree = CalibratedClassifierCV(base_tree, method = 'isotonic', cv = 5)
pipe = make_pipeline(preprocessor, calibrated_tree)
pipe.fit(X_train, y_train)


y_prob = pipe.predict_proba(X_test)[:, 1]
ece = calibration_error(y_test, y_prob, 'expected')
mce = calibration_error(y_test, y_prob, 'max')
bs = brier_score_loss(y_test, y_prob)
print(ece, mce, bs)

dump(pipe, 'swing.joblib')


0.05692527147639641 0.13879098811982274 0.187764345768603


['swing.joblib']